### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="consumer_complaints",
    dataset_year="2025",
    domain_str="finance",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://www.consumerfinance.gov/data-research/consumer-complaints/",
    download_description="""
We utilize the newest data (acquired on 23/01/2026) from the government website. We use the following commands to download and organize the data:

wget https://files.consumerfinance.gov/ccdb/complaints.csv.zip && unzip complaints.csv.zip && rm complaints.csv.zip
mkdir -p local-data-warehouse/consumer_complaints && mv complaints.csv local-data-warehouse/consumer_complaints
""",
    # References
    academic_reference_bibtex=r"""@misc{cfpb2025ConsumerComplaintDatabase,
  author       = {{Consumer Financial Protection Bureau}},
  title        = {Consumer Complaint Database},
  year         = {2025},
  howpublished = {\url{https://www.consumerfinance.gov/data-research/consumer-complaints/}},
  note         = {Accessed: 2026-01-23},
}
""",
    academic_reference_bibtex_key="cfpb2025ConsumerComplaintDatabase",
    license="U.S. Government Works",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
The dataset on Kaggle (https://www.kaggle.com/datasets/selener/consumer-complaint-database) has data from up until 2019. We use the newest version from the government website with data up until 2025.

- Context and descriptions of the features can be found here: https://cfpb.github.io/api/ccdb/fields.html
- "Company public response" is not free text but selected "from a set list of options".
- Following TexTabBench, we focus on predicting the type of closure a complaint got, that is the "Company response to consumer" column. We want to predict if a complaint will be closed with just an explanation, with non-monetary relief, or with monetary relief.
- We filter the data to only include entries after consumer disputations were discontinued as this represent a shift in protocol. This filters all data before April 24th 2017.
- We filter all cases where the "Consumer consent provided?" is in progress or got an untimely response.
- We filter all rows that do not include consent to share their complaint narrative. This ensures the data contains text sentences.
- We drop "Company public response" as it leaks the target variable.
- We only allow complaints from US states (no territories or international complaints) and remove cases with missing states.
- The data has unresolved spatial information in the ZIP code. Some ZIPs are censored (ending in "XXX" or full removed "XXXXXX").
- We add a feature for "Low population area", which determines that the ZIP is censored.
- The tags filed contains only three non-nan labels, of which one is a duplicate of the others. We created two categorical features from it instead.
- We drop duplicates (2% of the data) as the data should not contain naturally occurring duplicates and this likely results from some overlap in data collection or data entry or faulty re-submissions. We investigated some of the duplicates and they appear to be identical complaints. There exist duplicates with different target labels, we also drop these as we have no way to determine what the correct label is.
- We drop constant columns, the complaint ID, and when the complaint was send to the company (as it does not related to the target task)
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Company response to consumer",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Company response to consumer",
    time_on="Date received",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "complaints.csv")
print("Loaded data shape:", df.shape)

# Only keep entries after responses were discontinued as this represent a shift in protocol and closure (so data after April 2017)
# - Note this filters one of the classes from TexTabBench completely as it was an "old" label
df = df[df["Consumer disputed?"].isna()]
df = df.drop(columns=["Consumer disputed?"])

# Filter to complaints that include sentences and have consent
df = df[df["Consumer consent provided?"] == "Consent provided"]
df = df.drop(columns=["Consumer consent provided?"])

# Filter to only keep relevant rows that got closed
df = df[df["Company response to consumer"].isin(["Closed with explanation", "Closed with non-monetary relief", "Closed with monetary relief"])]

# Drop leakage variable
df = df.drop(columns=["Timely response?", "Company public response"])

# Remove non-us states
non_state = [
    "AA", "AE", "AP", "AS", "DC",
    "FM", "GU", "MH", "MP", "PR",
    "PW", "VI",
    "UNITED STATES MINOR OUTLYING ISLANDS",
]
df = df[~(df["State"].isin(non_state) | df["State"].isna())]

# if the ZIP ends with XXX, it means the complaints originates from a place with less than 20k people.
df["Low population area"] = "False"
df.loc[(df["ZIP code"].str.contains("XX") & (df["ZIP code"] != "XXXXX")), "Low population area"] = "True"
df.loc[df["ZIP code"] == "XXXXX", "Low population area"] = np.nan

# Resolve Multi-Tags feature into two categorical features
assert ['Older American', 'Older American, Servicemember', 'Servicemember', 'nan'] == list(np.unique(df["Tags"].astype(str)))
tags_nan_mask = df["Tags"].isna()
tags_is_older_american = df["Tags"].str.contains("Older American", na=False)
tags_is_servicemember = df["Tags"].str.contains("Servicemember", na=False)
df["Tag: Older American"] = "False"
df["Tag: Servicemember"] = "False"
df.loc[tags_is_older_american, "Tag: Older American"] = "True"
df.loc[tags_is_servicemember, "Tag: Servicemember"] = "True"
df.loc[tags_nan_mask, ["Tag: Older American", "Tag: Servicemember"]] = np.nan
df = df.drop(columns=["Tags"])

# Drop column
df = df.drop(columns=[
    "Submitted via", # only web after our preprocessing
    "Complaint ID", # we have new-ness of complaint from other columns, so this does not tell anything new
    "Date sent to company", # just shows processing delay from CFPB and not related to the target. Otherwise, almost always identical to "Date received".
])

as_cat_type = [
    "Product",
    # Higher cardinality categorical features, but still cat per definition of the data!
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Low population area",
    "Company response to consumer",
    "Tag: Older American",
    "Tag: Servicemember",
]
as_string_type = [
    "Company", # categorical based on feature description but given that we can have new companies in the future, it cannot be a "normal" categorical feature
    "Consumer complaint narrative",
    "ZIP code",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")
df["Date received"] = pd.to_datetime(df["Date received"])

# We drop duplicates as the data should not contain naturally occurring duplicates.
df = df.drop_duplicates(
    subset=df.columns.difference([task_mold.target_column_name])
)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
# Filter to year 2025
df_2025 = df[df[task_mold.time_on].dt.year == 2025].copy()

# Create a year-month column for grouping
df_2025["year_month"] = df_2025[task_mold.time_on].dt.to_period("M")

# 1) Total number of samples per month
monthly_totals = (
    df_2025
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# 2) Count of each class per month
monthly_class_counts = (
    df_2025
    .groupby(["year_month", task_mold.target_column_name])
    .size()
    .unstack(fill_value=0)
)

# Optional: combine both into a single DataFrame
result = monthly_class_counts.join(monthly_totals)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = result.sort_index()
result.index = result.index.to_timestamp()

In [ ]:
result

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_months = [
    "2025-02-01",
    "2025-03-01",
    "2025-04-01",
    "2025-05-01",
    "2025-06-01",
    "2025-07-01",
    "2025-08-01",
    "2025-09-01",
    "2025-10-01",
]
for i, month in enumerate(test_months):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year) &
        (df[task_mold.time_on].dt.month == ref_date.month)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

# Rest
ref_date  = pd.Timestamp("2025-11-01")
train_index = df[
    df[task_mold.time_on] < ref_date
].index
test_index = df[
    df[task_mold.time_on] >= ref_date
].index
splits[i + 1] = {
    0: (train_index.tolist(), test_index.tolist())
}

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

- The official data is updated daily but companies have 180 days to respond to a new complaint.
- We could simulate a model that is refitted daily, but this would need many splits. Moreover, data from just a few days is likely not enough to create a robust test set.
- Thus, we instead simulate a model that is refit every month and then deployed/refit until at the first of the next month.
- This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model.

We create ten test splits by using all months from 2025-02-01 to 2025-11-01 as test splits.
For each test split, we use all data before the test month as training data.

For the month 2025-11-01 we also include the small number of samples from 2025-12-01.
""",
    splits=splits,
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)